# Anima × ComfyUI inference diagnostics lab

학습 없이 Anima inference에서 **DiT block별 representation 통계, effective rank, spatial RMS heatmap, dtype/device**를 기록한다.

- ComfyUI와 Anima 모델 파일은 Google Drive가 아니라 **Colab 런타임에 직접 설치/다운로드**
- ComfyUI는 **Cloudflared URL**로 접속
- custom node: `Anima Inference Diagnostics`
- 결과: `/content/anima_diagnostics/<session>/`
- 마지막 셀에서 `@param`으로 지정한 Google Drive 경로로 이동

> 현재 heatmap은 Cosmos Predict2 `attn1_patch/attn2_patch`의 **Q/K/V projection 전 representation RMS map**이다. DAAM의 실제 token attention probability는 아니다.

In [ ]:
#@title 1. GPU 확인 + ComfyUI 설치 + diagnostic node 설치
import os, pathlib
!nvidia-smi

COMFY_DIR="/content/ComfyUI"
if not os.path.exists(COMFY_DIR):
    !git clone --depth 1 https://github.com/Comfy-Org/ComfyUI.git {COMFY_DIR}

%cd {COMFY_DIR}
!pip -q install -r requirements.txt
!pip -q install -U huggingface_hub pandas matplotlib pillow

node_dir=pathlib.Path(COMFY_DIR)/"custom_nodes/anima_inference_diagnostics"
node_dir.mkdir(parents=True,exist_ok=True)
!wget -q https://raw.githubusercontent.com/HisameOgasahara/deep-learning-diagnostics-and-improvement/main/Temp/anima_inference_diagnostics_node.py -O {node_dir}/__init__.py
print("custom node:",node_dir/"__init__.py")

In [ ]:
#@title 2. Anima Base + Qwen text encoder + VAE 다운로드
from huggingface_hub import hf_hub_download
from pathlib import Path
from PIL import Image
import os,json

repo="circlestone-labs/Anima"
comfy=Path("/content/ComfyUI")
files={
 "split_files/diffusion_models/anima-base-v1.0.safetensors":comfy/"models/diffusion_models/anima-base-v1.0.safetensors",
 "split_files/text_encoders/qwen_3_06b_base.safetensors":comfy/"models/text_encoders/qwen_3_06b_base.safetensors",
 "split_files/vae/qwen_image_vae.safetensors":comfy/"models/vae/qwen_image_vae.safetensors",
}
for remote,dst in files.items():
    dst.parent.mkdir(parents=True,exist_ok=True)
    if not dst.exists():
        cached=hf_hub_download(repo_id=repo,filename=remote)
        os.symlink(cached,dst)
    print(dst,f"{dst.stat().st_size/1024**3:.2f} GiB")

# 공식 model-card workflow PNG의 embedded workflow도 있으면 등록
png=hf_hub_download(repo_id=repo,filename="example.png")
img=Image.open(png)
wf=img.info.get("workflow")
if wf:
    wf=json.loads(wf) if isinstance(wf,str) else wf
    d=comfy/"user/default/workflows"; d.mkdir(parents=True,exist_ok=True)
    (d/"Anima_official_from_model_card.json").write_text(json.dumps(wf,ensure_ascii=False,indent=2),encoding="utf-8")
    print("workflow saved")

## ComfyUI에서 연결

공식 Anima workflow를 연 뒤 **sampler로 들어가는 마지막 MODEL 선 사이**에 `Anima Inference Diagnostics`를 끼운다.

기본값:
- blocks: `0,6,12,18,24,27` (Anima 2B는 28 blocks)
- scalar 기록: 매 model call
- spatial map + approximate effective rank: 5 model calls마다

처음엔 512×512 또는 768×768로 확인하는 편이 가볍다. 공식 Anima Base 권장 범위는 512²~1536², 30~50 steps, CFG 4~5다.

In [ ]:
#@title 3. ComfyUI + Cloudflared 실행
import os,re,time,subprocess,pathlib,signal
PORT=8188
for pf in ["/content/comfyui.pid","/content/cloudflared.pid"]:
    if os.path.exists(pf):
        try: os.kill(int(open(pf).read()),signal.SIGTERM)
        except: pass

cf="/content/cloudflared"
if not os.path.exists(cf):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O {cf}
    !chmod +x {cf}

log=open("/content/comfyui.log","w")
p=subprocess.Popen(["python","main.py","--listen","127.0.0.1","--port",str(PORT),"--lowvram"],cwd="/content/ComfyUI",stdout=log,stderr=subprocess.STDOUT)
open("/content/comfyui.pid","w").write(str(p.pid))
time.sleep(8)

cflog=open("/content/cloudflared.log","w")
q=subprocess.Popen([cf,"tunnel","--url",f"http://127.0.0.1:{PORT}","--no-autoupdate"],stdout=cflog,stderr=subprocess.STDOUT)
open("/content/cloudflared.pid","w").write(str(q.pid))

url=None
for _ in range(30):
    time.sleep(1)
    txt=pathlib.Path("/content/cloudflared.log").read_text(errors="ignore")
    m=re.search(r"https://[-a-z0-9]+\.trycloudflare\.com",txt)
    if m: url=m.group(0); break
print("OPEN:",url)

In [ ]:
#@title 4. 최신 diagnostic session을 CSV/그래프로 정리
!wget -q https://raw.githubusercontent.com/HisameOgasahara/deep-learning-diagnostics-and-improvement/main/Temp/postprocess_anima_diagnostics.py -O /content/postprocess_anima_diagnostics.py
%run /content/postprocess_anima_diagnostics.py

In [ ]:
#@title 5. 결과 미리보기
from pathlib import Path
from IPython.display import display,Image
root=Path("/content/anima_diagnostics")
ss=sorted([p for p in root.glob("*") if p.is_dir()],key=lambda p:p.stat().st_mtime)
if ss:
    latest=ss[-1]
    print(latest)
    for n in ["model_input_rms_by_call.png","attn1_q_rms_block_call_heatmap.png","effective_rank_by_call.png"]:
        p=latest/n
        if p.exists(): display(Image(filename=str(p)))

In [ ]:
#@title 6. @param Google Drive 경로로 결과 이동
from google.colab import drive
from pathlib import Path
import shutil,time
drive.mount("/content/drive")

drive_destination="/content/drive/MyDrive/AnimaDiagnostics" #@param {type:"string"}
move_instead_of_copy=True #@param {type:"boolean"}
also_transfer_comfyui_images=True #@param {type:"boolean"}

root=Path("/content/anima_diagnostics")
ss=sorted([p for p in root.glob("*") if p.is_dir()],key=lambda p:p.stat().st_mtime)
if not ss: raise RuntimeError("diagnostic session 없음")
src=ss[-1]; dstroot=Path(drive_destination); dstroot.mkdir(parents=True,exist_ok=True)
dst=dstroot/src.name
if dst.exists(): dst=dstroot/f"{src.name}_{int(time.time())}"
(shutil.move(str(src),str(dst)) if move_instead_of_copy else shutil.copytree(src,dst))
if also_transfer_comfyui_images:
    out=Path("/content/ComfyUI/output"); target=dst/"comfyui_output"; target.mkdir(parents=True,exist_ok=True)
    if out.exists():
        for p in out.rglob("*"):
            if p.is_file():
                t=target/p.relative_to(out); t.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(p,t)
print("saved to:",dst)

## 첫 관찰 포인트
- `model_input_rms_by_call.png`: sigma가 내려가며 latent/model input scale이 어떻게 변하는가
- `attn1_q_rms_block_call_heatmap.png`: 어느 DiT block이 어느 inference 시점에서 크게 변하는가
- `effective_rank_by_call.png`: 선택 block의 representation이 실질적으로 몇 방향을 쓰는가
- `maps/*.png`: 공간 위치별 representation RMS
- `dtype_device_table.csv`: image/text 경로의 dtype/device 불일치 여부

다음 단계에서 projected Q/K를 계측하면 DAAM류 text-token spatial attention map으로 확장할 수 있다.